# R3 Contamination diagnostics (E3) on Qwen2.5-3B: saved outputs

Before reusing a public dataset, we test whether the model may have seen it in training. Two checks: **guided completion** (give the first half of an item and the dataset name; does the model reproduce the rest nearly word for word?) and **Min-K% Prob** (does the original text look less surprising to the model than a paraphrase of it?). A source is flagged if more than 5% of items come back near verbatim, or originals beat paraphrases in more than 70% of pairs.

These are diagnostics. A clean result does not prove the data was never seen.

In [1]:
# Works locally (run from the repo) and in Colab (clones the public repo; no GPU needed).
import os, json, subprocess
if not os.path.exists('results/behaviour'):
    if not os.path.exists('/content/ContextLLM'):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/Tanjamul-Azad/ContextLLM.git', '/content/ContextLLM'], check=True)
    os.chdir('/content/ContextLLM')
import pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)
from IPython.display import Image, Markdown, display
print('reading saved results from', os.path.abspath('results'))

reading saved results from F:\UIU\13\Paper\ContextLLM\results


## GSM8K (first run, 60 questions)

In [2]:
print(json.dumps(json.load(open('results/contamination/qwen2.5-3b/summary.json')), indent=1))

{
 "model": "Qwen/Qwen2.5-3B-Instruct",
 "dataset": "openai/gsm8k test, 60 questions from offset 487",
 "guided_completion": {
  "mean_rouge_l_guided": 0.278,
  "mean_rouge_l_general": 0.305,
  "share_near_verbatim_guided(>=0.75)": 0.0,
  "share_near_verbatim_general(>=0.75)": 0.017
 },
 "min_k_prob(k=20%)": {
  "mean_original": -8.919,
  "mean_paraphrase": -7.004,
  "share_original_scores_higher_than_its_paraphrase": 0.167,
  "mean_our_fresh_seed_texts": -9.418
 },
 "rule": "Flag the source if more than 5% of items come back near word for word, or if originals score clearly higher than paraphrases.",
 "flag_source": false
}


In [3]:
g = pd.read_csv('results/contamination/qwen2.5-3b/items.csv')
display(g.describe().round(3))

,rouge_guided,rouge_general,mink_original,mink_paraphrase
count,60.000,60.000,60.000,60.000
mean,0.278,0.305,-8.919,-7.004
std,0.139,0.154,2.017,1.698
min,0.077,0.000,-14.242,-12.008
25%,0.159,0.192,-10.190,-8.002
50%,0.278,0.273,-8.850,-6.710
75%,0.362,0.436,-7.843,-5.765
max,0.667,0.769,-4.714,-4.135


## Candidate training sources (50 items each)

In [4]:
display(Markdown(open('results/contamination/qwen2.5-3b/sources/README.md', encoding='utf-8').read()))

# Contamination checks on reused public sources (Qwen2.5-3B-Instruct)

Script: `src/contamination_sources.py` (run 2026-11-05, 02:25 to 02:44, about 19 minutes on one RTX 4060; log in `results/logs/contamination_sources.log`).
Same two checks as the GSM8K run in `results/contamination/qwen2.5-3b/summary.json`:

- Guided completion (Golchin and Surdeanu, ICLR 2024): the model gets the first half of an item, with the dataset name (guided) or without it (general). Near-verbatim = ROUGE-L >= 0.75 against the real second half.
- Min-K% Prob, k = 20% (Shi et al., ICLR 2024): each original is scored against a paraphrase written by the same model. The column gives the share of pairs where the original scores higher.

Flag rule (protocol section 3): flag if guided near-verbatim share > 5%, or originals beat paraphrases in > 70% of pairs.

| Source | Items used | n | Near-verbatim guided / general | Mean ROUGE-L guided / general | Min-K% original > paraphrase | Flagged |
| --- | --- | ---: | ---: | ---: | ---: | --- |
| Super-NaturalInstructions | task definitions, `jayelm/natural-instructions` test split, 50 distinct tasks | 50 | 0.00 / 0.00 | 0.188 / 0.193 | 0.70 | no (at the threshold) |
| HumanEval | function prompts, `openai/openai_humaneval` test | 50 | 0.00 / 0.00 | 0.157 / 0.171 | 0.40 | no |
| Spider | questions with >= 12 words, `xlangai/spider` validation | 50 | 0.00 / 0.00 | 0.133 / 0.096 | 0.36 | no |
| AnaloBench | gold stories, `jhu-clsp/AnaloBench` config T1S10-Full | 50 | 0.00 / 0.00 | 0.145 / 0.150 | 0.36 | no |
| ParallelPARC | - | 0 | - | - | - | not tested |
| GSM8K (earlier run, for reference) | test questions | 60 | 0.00 / 0.017 | 0.278 / 0.305 | 0.17 | no |

Per-item scores are in `<source>/items.csv`. These files hold item ids (task name, row index or task id) and scores only, with no dataset text. Run status is in `run_status.json`.

## Notes

- Super-NaturalInstructions is borderline. Originals beat paraphrases in exactly 70% of pairs, and the mean Min-K% gap is the largest of the four sources (-5.97 vs -7.03). Strictly, the rule (> 70%) does not flag it. These task definitions are widely used in instruction tuning (Flan, Tulu and others), so exposure is plausible. Treat reused definitions as at risk: rewrite them before they enter test, or recheck them with more items and a second model.
- ParallelPARC was skipped. On 2026-11-05 a Hub search for "ParallelPARC", "parallelparc", "parallel_parc" and "Parallel-PARC" returned no datasets. It was not fetched from any other host.
- `Muennighoff/natural-instructions` was not usable because the dataset viewer still reported it as processing. The public mirror `jayelm/natural-instructions` was used instead. Super-NI was tested only on its task definitions, not on positive or negative example text.
- Spider questions are short (median about 15 words), so each half is only 6 to 10 words and ROUGE-L is noisy. The Spider row covers the questions only, not the SQL.
- The originals of every source score higher (less surprising) than our own seed texts (-9.42), except Spider (-10.13). That comparison mixes text types and is not evidence on its own.

## Limitations

- Only one small model was tested (Qwen2.5-3B-Instruct, greedy decoding). Larger or other models may memorise more.
- A negative result does not prove an item was absent from training data. Guided completion misses items the model saw but cannot reproduce. Min-K% against self-written paraphrases is weak when paraphrases are themselves fluent, high-probability text.
- 50 items per source give a wide interval. A 0% near-verbatim share is consistent with a true rate of a few percent.
- These checks cover the public source texts only. They do not certify new items we build from those sources.


In [5]:
import glob
for f in sorted(glob.glob('results/contamination/qwen2.5-3b/sources/*/items.csv')):
    d = pd.read_csv(f); print(f.split(os.sep)[-2] if os.sep in f else f, d.shape); display(d.head(5))

analobench (50, 6)


,item_id,n_words,rouge_guided,rouge_general,mink_original,mink_paraphrase
0,T1S10-Full|row1326|optD,193,0.141,0.107,-5.942,-6.556
1,T1S10-Full|row617|optB,163,0.161,0.138,-6.667,-5.964
2,T1S10-Full|row1617|optC,176,0.128,0.115,-5.726,-6.308
3,T1S10-Full|row197|optC,171,0.139,0.112,-6.509,-6.477
4,T1S10-Full|row296|optC,177,0.161,0.157,-6.045,-6.498


humaneval (50, 6)


,item_id,n_words,rouge_guided,rouge_general,mink_original,mink_paraphrase
0,HumanEval/7,39,0.115,0.086,-3.516,-3.535
1,HumanEval/8,62,0.071,0.120,-3.852,-3.264
2,HumanEval/9,47,0.086,0.224,-4.570,-4.533
3,HumanEval/11,43,0.149,0.120,-4.236,-4.643
4,HumanEval/12,55,0.167,0.162,-3.807,-2.717


natural_instructions (50, 6)


,item_id,n_words,rouge_guided,rouge_general,mink_original,mink_paraphrase
0,task402_grailqa_paraphrase_generation|row339563,133,0.211,0.211,-11.151,-10.403
1,task1533_daily_dialog_formal_classification|row158176,52,0.215,0.142,-7.613,-7.002
2,task620_ohsumed_medical_subject_headings_answer_generation|row414002,31,0.111,0.190,-10.300,-6.998
3,task1631_openpi_answer_generation|row682554,33,0.161,0.149,-6.453,-8.445
4,task102_commongen_sentence_generation|row50631,62,0.129,0.233,-4.794,-5.785


spider (50, 6)


,item_id,n_words,rouge_guided,rouge_general,mink_original,mink_paraphrase
0,pets_1|row62,20,0.083,0.074,-9.000,-8.097
1,pets_1|row76,15,0.091,0.091,-9.598,-8.793
2,pets_1|row78,16,0.250,0.190,-11.424,-13.866
3,pets_1|row79,15,0.000,0.000,-10.212,-7.925
4,car_1|row90,13,0.250,0.205,-9.962,-8.853


In [6]:
# Optional, Colab only: copy these saved outputs into Google Drive (DL paper/03_Results/contamination).
import shutil
try:
    from google.colab import drive
    drive.mount('/content/drive')
    for src in ['results/contamination']:
        dst = '/content/drive/MyDrive/DL paper/03_Results/contamination/' + os.path.basename(src.rstrip('/'))
        shutil.copytree(src, dst, dirs_exist_ok=True); print('copied', src, '->', dst)
except ImportError:
    print('Not in Colab: skipped. The files are in the repo under', ['results/contamination'])

Not in Colab: skipped. The files are in the repo under ['results/contamination']
